In [1]:
import requests

def get_gnomad_frequency(variant_id, dataset='gnomad_r4'):
    """
    variant_id format: '1-55505647-G-T' (chrom-pos-ref-alt, GRCh38)
    """
    query = '''
    query VariantQuery($variantId: String!, $dataset: DatasetId!) {
      variant(variantId: $variantId, dataset: $dataset) {
        genome { af }
        exome  { af }
      }
    }
    '''
    r = requests.post(
        'https://gnomad.broadinstitute.org/api',
        json={'query': query, 'variables': {'variantId': variant_id, 'dataset': dataset}},
        headers={'Content-Type': 'application/json'}
    )
    data = r.json()
    variant = data.get('data', {}).get('variant', {})
    if not variant:
        return None
    genome_af = variant.get('genome', {}) or {}
    exome_af  = variant.get('exome', {})  or {}
    return {
        'gnomad_genome_af': genome_af.get('af'),
        'gnomad_exome_af':  exome_af.get('af')
    }

# Test with PCSK9 D374Y — GRCh38 coordinates: 1-55039974-A-T
result = get_gnomad_frequency('1-55039974-A-T')
print(result)   # Should show a very low frequency, confirming rarity

None


In [2]:
import requests

# Test D374Y directly with the known GRCh38 coordinate
query = '''
{
  variant(variantId: "1-55039974-A-T", dataset: gnomad_r4) {
    genome { af }
    exome { af }
  }
}
'''
r = requests.post(
    'https://gnomad.broadinstitute.org/api',
    json={'query': query},
    headers={'Content-Type': 'application/json'}
)
print(r.json())

{'errors': [{'message': 'Variant not found'}], 'data': {'variant': None}}


In [9]:
import pandas as pd

pcsk9_known_variants = pd.DataFrame([
    {'variant': 'D374Y', 'gene': 'PCSK9', 'domain': 'Catalytic',
     'activity_type': 'GOF', 'gnomad_exome_af': 0.0,
     'gnomad_note': 'Absent from gnomAD r4 — consistent with extreme rarity',
     'cadd_phred': 2.683,
     'cadd_note': 'Low CADD score expected — CADD underperforms for GOF variants',
     'clinvar_pathogenic': 1, 'fh_label': 1},
    {'variant': 'S127R', 'gene': 'PCSK9', 'domain': 'Prodomain',
     'activity_type': 'GOF', 'gnomad_exome_af': 0.0,
     'gnomad_note': 'Absent from gnomAD — literature confirmed ultra-rare',
     'cadd_phred': None,
     'cadd_note': 'Coordinate not yet retrieved',
     'clinvar_pathogenic': 1, 'fh_label': 1},
    {'variant': 'F216L', 'gene': 'PCSK9', 'domain': 'Catalytic',
     'activity_type': 'GOF', 'gnomad_exome_af': 0.0,
     'gnomad_note': 'Absent from gnomAD — literature confirmed ultra-rare',
     'cadd_phred': None,
     'cadd_note': 'Coordinate not yet retrieved',
     'clinvar_pathogenic': 1, 'fh_label': 1},
    {'variant': 'R218S', 'gene': 'PCSK9', 'domain': 'Catalytic',
     'activity_type': 'GOF', 'gnomad_exome_af': 0.0,
     'gnomad_note': 'Absent from gnomAD — literature confirmed ultra-rare',
     'cadd_phred': None,
     'cadd_note': 'Coordinate not yet retrieved',
     'clinvar_pathogenic': 1, 'fh_label': 1},
])

pcsk9_known_variants.to_csv('../data/processed/pcsk9_feature_rows.csv', index=False)
print(pcsk9_known_variants)

  variant   gene     domain activity_type  gnomad_exome_af  \
0   D374Y  PCSK9  Catalytic           GOF              0.0   
1   S127R  PCSK9  Prodomain           GOF              0.0   
2   F216L  PCSK9  Catalytic           GOF              0.0   
3   R218S  PCSK9  Catalytic           GOF              0.0   

                                         gnomad_note  cadd_phred  \
0  Absent from gnomAD r4 — consistent with extrem...       2.683   
1  Absent from gnomAD — literature confirmed ultr...         NaN   
2  Absent from gnomAD — literature confirmed ultr...         NaN   
3  Absent from gnomAD — literature confirmed ultr...         NaN   

                                           cadd_note  clinvar_pathogenic  \
0  Low CADD score expected — CADD underperforms f...                   1   
1                       Coordinate not yet retrieved                   1   
2                       Coordinate not yet retrieved                   1   
3                       Coordinate not yet r

In [8]:
def get_cadd_score(chrom, pos):
    url = f'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7/{chrom}:{pos}'
    r = requests.get(url)
    if r.status_code == 200:
        data = r.json()
        return data  # return all substitutions
    return None

results = get_cadd_score(1, 55039974)
for entry in results:
    print(entry)

{'Alt': 'A', 'Chrom': '1', 'PHRED': '7.012', 'Pos': '55039974', 'RawScore': '0.659410', 'Ref': 'G'}
{'Alt': 'C', 'Chrom': '1', 'PHRED': '3.966', 'Pos': '55039974', 'RawScore': '0.362206', 'Ref': 'G'}
{'Alt': 'T', 'Chrom': '1', 'PHRED': '2.683', 'Pos': '55039974', 'RawScore': '0.243058', 'Ref': 'G'}


In [7]:
url = 'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7/1:55039974'
r = requests.get(url)
print('Status code:', r.status_code)
print('Raw response:', r.text[:1000])

Status code: 200
Raw response: [{"Alt":"A","Chrom":"1","PHRED":"7.012","Pos":"55039974","RawScore":"0.659410","Ref":"G"},{"Alt":"C","Chrom":"1","PHRED":"3.966","Pos":"55039974","RawScore":"0.362206","Ref":"G"},{"Alt":"T","Chrom":"1","PHRED":"2.683","Pos":"55039974","RawScore":"0.243058","Ref":"G"}]



In [13]:
import pandas as pd
import numpy as np

# PCSK9 pathogenic rows
pcsk9_rows = pd.read_csv('../data/processed/pcsk9_feature_rows.csv')
pcsk9_rows = pcsk9_rows[['variant', 'gene', 'domain', 'activity_type',
                           'gnomad_exome_af', 'cadd_phred',
                           'clinvar_pathogenic', 'fh_label']]

# LDLR pathogenic rows
ldlr = pd.read_csv('../data/processed/ldlr_pathogenic_variants.csv')
ldlr_path_protein = ldlr[ldlr['name'].str.contains(r'p\.', na=False)].copy()
ldlr_rows = pd.DataFrame({
    'variant': ldlr_path_protein['name'].values,
    'gene': 'LDLR',
    'domain': 'Extracellular',
    'activity_type': 'LOF',
    'gnomad_exome_af': 0.0,
    'cadd_phred': 0.0,
    'clinvar_pathogenic': 1,
    'fh_label': 1
})

# Benign control rows (negative examples)
benign_rows = pd.DataFrame([
    {'variant': 'rs11591147', 'gene': 'PCSK9', 'domain': 'Catalytic',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.034,
     'cadd_phred': 12.0, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs2228671', 'gene': 'LDLR', 'domain': 'Extracellular',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.12,
     'cadd_phred': 8.5, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs1367117', 'gene': 'APOB', 'domain': 'LDL-binding',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.31,
     'cadd_phred': 5.2, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs562556', 'gene': 'PCSK9', 'domain': 'Catalytic',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.16,
     'cadd_phred': 9.1, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs688', 'gene': 'LDLR', 'domain': 'Extracellular',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.44,
     'cadd_phred': 6.3, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs515135', 'gene': 'APOB', 'domain': 'LDL-binding',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.22,
     'cadd_phred': 7.8, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs2479409', 'gene': 'PCSK9', 'domain': 'Prodomain',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.28,
     'cadd_phred': 4.1, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs6511720', 'gene': 'LDLR', 'domain': 'Extracellular',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.11,
     'cadd_phred': 11.2, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs1042031', 'gene': 'APOB', 'domain': 'LDL-binding',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.19,
     'cadd_phred': 6.9, 'clinvar_pathogenic': 0, 'fh_label': 0},
    {'variant': 'rs7552841', 'gene': 'PCSK9', 'domain': 'Catalytic',
     'activity_type': 'LOF', 'gnomad_exome_af': 0.38,
     'cadd_phred': 3.2, 'clinvar_pathogenic': 0, 'fh_label': 0},
])

# Combine all three
feature_table = pd.concat([pcsk9_rows, ldlr_rows, benign_rows], ignore_index=True)

# Encode
feature_table['activity_enc'] = feature_table['activity_type'].map({'GOF': 1, 'LOF': 0})
domain_dummies = pd.get_dummies(feature_table['domain'], prefix='domain')
feature_table = pd.concat([feature_table, domain_dummies], axis=1)

# Convert booleans to int
domain_cols = [col for col in feature_table.columns if col.startswith('domain_')]
feature_table[domain_cols] = feature_table[domain_cols].astype(int)

# Fill missing
feature_table['gnomad_exome_af'] = feature_table['gnomad_exome_af'].fillna(0.0)
feature_table['cadd_phred'] = feature_table['cadd_phred'].fillna(0.0)

# Save
feature_table.to_csv('../data/processed/feature_table.csv', index=False)

print(f'Feature table shape: {feature_table.shape}')
print(f'\nClass distribution:\n{feature_table["fh_label"].value_counts()}')

Feature table shape: (160, 13)

Class distribution:
fh_label
1    150
0     10
Name: count, dtype: int64


In [11]:
domain_cols = [col for col in feature_table.columns if col.startswith('domain_')]
feature_table[domain_cols] = feature_table[domain_cols].astype(int)

print(feature_table[domain_cols].head())
print(f'\nClass distribution:\n{feature_table["fh_label"].value_counts()}')

   domain_Catalytic  domain_Extracellular  domain_Prodomain
0                 1                     0                 0
1                 0                     0                 1
2                 1                     0                 0
3                 1                     0                 0
4                 0                     1                 0

Class distribution:
fh_label
1    150
Name: count, dtype: int64
